# PRE EDA

## Разбор датасета Open Targets

Источник: [Open Targets Platform 26.09](https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/26.09/output/).

## 1. Препараты и клиника

Исходные Parquet-файлы сохраняются в `data/<dataset>/`, а первые 100 строк каждого набора — в `samples/<dataset>_sample.csv`. Повторный запуск использует уже скачанные файлы.

In [12]:
%pip install pandas pyarrow requests tqdm

5409.84s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
import re
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

RELEASE = "26.09"
BASE_URL = f"https://ftp.ebi.ac.uk/pub/databases/opentargets/platform/{RELEASE}/output/"
SAMPLE_ROWS = 100

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

DATA_DIR = project_root / "data"
SAMPLES_DIR = project_root / "samples"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)


session = requests.Session()
retries = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET",),
)
session.mount("https://", HTTPAdapter(max_retries=retries))
session.headers.update({"User-Agent": "open-targets-pre-eda/1.0"})


def _get_url(url, stream=False):
    response = session.get(url, stream=stream, timeout=(15, 180))
    response.raise_for_status()
    return response


def download_dataset(dataset_name):
    """Скачать все Parquet-части датасета и вернуть локальные пути."""
    dataset_url = urljoin(BASE_URL, f"{dataset_name}/")
    with _get_url(dataset_url) as response:
        directory_html = response.text

    parquet_links = sorted(set(re.findall(r'href="([^"]+\.parquet)"', directory_html)))
    if not parquet_links:
        raise FileNotFoundError(f"В каталоге {dataset_url} не найдены Parquet-файлы")

    dataset_dir = DATA_DIR / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)
    local_paths = []

    for file_index, parquet_link in enumerate(parquet_links, start=1):
        file_url = urljoin(dataset_url, parquet_link)
        file_name = Path(urlparse(file_url).path).name
        destination = dataset_dir / file_name

        if not destination.exists():
            temporary = destination.with_suffix(f"{destination.suffix}.part")
            with _get_url(file_url, stream=True) as response, temporary.open("wb") as output:
                total_bytes = int(response.headers.get("content-length", 0))
                description = f"{dataset_name} [{file_index}/{len(parquet_links)}] {file_name}"
                with tqdm(
                    total=total_bytes or None,
                    desc=description,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        output.write(chunk)
                        progress.update(len(chunk))
            temporary.replace(destination)

        local_paths.append(destination)

    return local_paths


def load_dataset_sample(dataset_name, sample_rows=SAMPLE_ROWS):
    """Скачать датасет, прочитать первые строки и сохранить их в CSV."""
    parquet_paths = download_dataset(dataset_name)
    total_rows = sum(pq.ParquetFile(path).metadata.num_rows for path in parquet_paths)

    frames = []
    rows_left = min(sample_rows, total_rows)
    for path in parquet_paths:
        parquet_file = pq.ParquetFile(path)
        for batch in parquet_file.iter_batches(batch_size=rows_left):
            batch_df = batch.to_pandas()
            frames.append(batch_df)
            rows_left -= len(batch_df)
            if rows_left == 0:
                break
        if rows_left == 0:
            break

    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    sample_path = SAMPLES_DIR / f"{dataset_name}_sample.csv"
    df.to_csv(sample_path, index=False)

    print(f"{dataset_name}: {total_rows:,} строк, {len(df.columns)} столбцов")
    print(f"CSV-сэмпл: {sample_path.relative_to(project_root)} ({len(df)} строк)")
    return df

In [18]:
disease_df = load_dataset_sample("disease")

disease [1/1] 00000000.parquet: 100%|██████████| 6.05M/6.05M [00:28<00:00, 223kB/s]

disease: 45,896 строк, 18 столбцов
CSV-сэмпл: samples/disease_sample.csv (100 строк)


In [21]:
disease_df.iloc[99]

id                                                         GO_0007265
code                        http://purl.obolibrary.org/obo/GO_0007265
name                                  Ras protein signal transduction
description         An intracellular signaling cassette in which a...
dbXRefs               [Reactome:R-HSA-1169092, Reactome:R-HSA-167044]
parents                                                  [GO_0007165]
exactSynonyms                      [Ras mediated signal transduction]
relatedSynonyms                                                    []
narrowSynonyms                                                     []
broadSynonyms                                                      []
obsoleteTerms                                                      []
obsoleteXRefs                                                      []
children                                                           []
ancestors           [GO_0007165, GO_0009987, GO_0050794, GO_000815...
therapeuticAreas    

In [22]:
drug_molecule_df = load_dataset_sample("drug_molecule")

drug_molecule [1/1] 00000000.parquet: 100%|██████████| 6.55M/6.55M [00:09<00:00, 753kB/s]

drug_molecule: 19,170 строк, 13 столбцов
CSV-сэмпл: samples/drug_molecule_sample.csv (100 строк)
